# Model Results

Fits and evalutes the Bayesian model.

## Imports

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch

In [2]:
# Find root directory of project
root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "config.py").exists())
sys.path.insert(0, str(root))

In [3]:
from src.config import PROCESSED_DATA_DIR, RANDOM_SEED, TABLES_DIR, TRAIN_SEASONS, TEST_SEASONS
from src.model.model import BayesianPoissonModel, BayesianPoissonModelConfig, SVIConfig

/Users/astonchan/Desktop/Academics Folder/UC Irvine/Senior Year/Spring Quarter/COMPSCI 179 – Algorithms for Probabilistic and Deterministic Graphical Models/Project/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Evaluation

### Load Training and Testing Data

In [4]:
df = pd.read_csv(PROCESSED_DATA_DIR / "epl_matches.csv", parse_dates = ["Date"])

train_df = df[df["Season"].isin(TRAIN_SEASONS)].reset_index(drop = True)
test_df = df[df["Season"].isin(TEST_SEASONS)].reset_index(drop = True)

print(f"Train: {len(train_df)} matches")
print(f"Test: {len(test_df)} matches")

Train: 3040 matches
Test: 760 matches


### Bayesian Model

In [5]:
bayesian_model = BayesianPoissonModel(
    config = BayesianPoissonModelConfig(
        seed = RANDOM_SEED,
        device = (torch.device("cuda")
                  if torch.cuda.is_available()
                  else torch.device("cpu")),
        svi = SVIConfig(),
        ablate_attack = False,
        ablate_defense = False,
        ablate_home_team_advantage = False,
        num_posterior_samples_per_match = 100,
    )
).fit(train_df)
bayesian_model_preds = bayesian_model.predict(test_df)

Step 100: Loss = 9359.277
Step 200: Loss = 9192.884
Step 300: Loss = 9184.015
Step 400: Loss = 9190.682
Step 500: Loss = 9169.601
Step 600: Loss = 9168.463
Step 700: Loss = 9171.383
Step 800: Loss = 9167.521
Step 900: Loss = 9169.867
Step 1000: Loss = 9167.339
Step 1100: Loss = 9167.054
Step 1200: Loss = 9170.397
Step 1300: Loss = 9167.762
Step 1400: Loss = 9165.760
Step 1500: Loss = 9168.367
Step 1600: Loss = 9167.892
Step 1700: Loss = 9165.212
Step 1800: Loss = 9170.267
Step 1900: Loss = 9168.235
Step 2000: Loss = 9168.364
Step 2100: Loss = 9167.400
Step 2200: Loss = 9168.218
Step 2300: Loss = 9166.698
Step 2400: Loss = 9167.051
Step 2500: Loss = 9166.675
Step 2600: Loss = 9169.116
Step 2700: Loss = 9168.019
Step 2800: Loss = 9169.710
Step 2900: Loss = 9167.021
Step 3000: Loss = 9169.175
Step 3100: Loss = 9167.444
Step 3200: Loss = 9164.598
Step 3300: Loss = 9166.560
Step 3400: Loss = 9165.710
Step 3500: Loss = 9166.202
Step 3600: Loss = 9164.445
Step 3700: Loss = 9168.148
Step 3800:

In [6]:
bayesian_model_preds[["HomeTeam", "AwayTeam", "Result", "ProbHomeWin", "ProbDraw", "ProbAwayWin"]].head()

,HomeTeam,AwayTeam,Result,ProbHomeWin,ProbDraw,ProbAwayWin
0,Man United,Fulham,H,0.623498,0.215093,0.161409
1,West Ham,Aston Villa,A,0.372980,0.256246,0.370774
2,Nott'm Forest,Bournemouth,D,0.536085,0.224819,0.239097
3,Newcastle,Southampton,H,0.544045,0.235536,0.220419
4,Everton,Brighton,A,0.434198,0.275700,0.290102


In [7]:
pred_labels = bayesian_model_preds[["ProbHomeWin",
                                    "ProbDraw",
                                    "ProbAwayWin"]].idxmax(axis = 1).map({"ProbHomeWin": "H",
                                                                          "ProbDraw": "D",
                                                                          "ProbAwayWin": "A"})
accuracy = (pred_labels == bayesian_model_preds["Result"]).mean()
print(f"Bayesian Model Accuracy: {accuracy:.3f}")

Bayesian Model Accuracy: 0.451


### Save Evaluation Results

In [8]:
model_table_output_directory = (TABLES_DIR / "model")
model_table_output_directory.mkdir(parents = True, exist_ok = True)

bayesian_model_preds.to_csv(model_table_output_directory / "bayesian_model_predictions.csv",
                            index = False)